# 导入包 + 参数

In [ ]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from tqdm.auto import tqdm
from IPython.display import display


# 路径
INPUT_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/anno_h5ads"
)

OUT_DIR = Path(
    "/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/spot_level_DE_outer_min10"
)

PER_REGION_DIR = OUT_DIR / "per_region"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PER_REGION_DIR.mkdir(parents=True, exist_ok=True)


# 样本
GROUP_MAP = {
    "G21-4": "Young",
    "G21-7": "Young",
    "G21-1": "Aged",
    "G21-8": "Aged",
    "G21-2": "AD",
    "G21-5": "AD",
}

SAMPLES = list(GROUP_MAP.keys())


# 脑区
REGIONS = [
    "CTX_1", "CTX_2", "CTX_3",
    "HIP", "CA1", "CA3", "DG",
    "TH", "HY", "STR", "FT", "VS",
]


# 比较
CONTRASTS = {
    "Aged_vs_Young": ("Aged", "Young"),
    "AD_vs_Young": ("AD", "Young"),
    "Aged_vs_AD": ("Aged", "AD"),
}


# 参数
REGION_KEY = "Regions"
COUNT_LAYER = "counts"
SYMBOL_KEY = "Symbol"

MIN_DETECTED_SPOTS = 10
MIN_GROUP_SPOTS = 30

print("Output:", OUT_DIR)

# 读取6个样本

In [ ]:
adata_list = []
annotation_list = []

for sample in tqdm(SAMPLES, desc="Reading samples"):
    path = INPUT_DIR / f"{sample}_regions.h5ad"
    ad = sc.read_h5ad(path)

    if REGION_KEY not in ad.obs.columns:
        raise KeyError(f"{sample}: 缺少 obs['{REGION_KEY}']")
    if COUNT_LAYER not in ad.layers:
        raise KeyError(f"{sample}: 缺少 layers['{COUNT_LAYER}']")
    if not ad.var_names.is_unique:
        raise ValueError(f"{sample}: var_names 不唯一")

    # gene annotation
    symbol = ad.var[SYMBOL_KEY].astype("string") if SYMBOL_KEY in ad.var.columns else ad.var_names.astype(str)

    annotation_list.append(
        pd.DataFrame({
            "gene_id": ad.var_names.astype(str),
            "Symbol": np.asarray(symbol),
        })
    )

    # 只保留目标脑区
    ad.obs[REGION_KEY] = ad.obs[REGION_KEY].astype(str)
    ad = ad[ad.obs[REGION_KEY].isin(REGIONS)].copy()

    ad.obs["sample_id"] = sample
    ad.obs["condition"] = GROUP_MAP[sample]
    ad.obs_names = sample + "_" + ad.obs_names.astype(str)

    ad.layers[COUNT_LAYER] = sp.csr_matrix(ad.layers[COUNT_LAYER])
    adata_list.append(ad)

    print(f"{sample} | {GROUP_MAP[sample]} | {ad.n_obs:,} spots | {ad.n_vars:,} genes")

# 整理 annotation + outer 合并

In [ ]:
# 整理gene annotation
gene_annotation = pd.concat(annotation_list, ignore_index=True)
gene_annotation["Symbol"] = gene_annotation["Symbol"].astype("string")

missing = gene_annotation["Symbol"].isna() | gene_annotation["Symbol"].eq("")
gene_annotation.loc[missing, "Symbol"] = gene_annotation.loc[missing, "gene_id"]

# 检查一个gene_id是否对应多个Symbol
conflict = gene_annotation.groupby("gene_id")["Symbol"].nunique()
conflict = conflict[conflict > 1]

print("存在多个Symbol映射的gene_id数量：", len(conflict))

if len(conflict) > 0:
    display(
        gene_annotation[gene_annotation["gene_id"].isin(conflict.index)]
        .sort_values("gene_id")
        .head(50)
    )

gene_annotation = (
    gene_annotation
    .drop_duplicates(["gene_id", "Symbol"])
    .drop_duplicates("gene_id")
    .set_index("gene_id")
)


# outer合并
adata_spot = sc.concat(
    adata_list,
    join="outer",
    merge="first",
    uns_merge="first",
    fill_value=0,
)

adata_spot.layers[COUNT_LAYER] = sp.csr_matrix(adata_spot.layers[COUNT_LAYER])

# 重新补充Symbol
symbol_map = gene_annotation["Symbol"].to_dict()
adata_spot.var[SYMBOL_KEY] = adata_spot.var_names.astype(str).map(symbol_map)

missing = adata_spot.var[SYMBOL_KEY].isna() | adata_spot.var[SYMBOL_KEY].eq("")
adata_spot.var.loc[missing, SYMBOL_KEY] = adata_spot.var_names[missing].astype(str)

adata_spot.obs["condition"] = pd.Categorical(
    adata_spot.obs["condition"].astype(str),
    categories=["Young", "Aged", "AD"],
)

del adata_list, annotation_list
gc.collect()

print(adata_spot)
print("Union genes:", adata_spot.n_vars)

# 检查各样本、各脑区 spot 数

In [ ]:
spot_counts_sample = pd.crosstab(
    adata_spot.obs["sample_id"],
    adata_spot.obs[REGION_KEY],
).reindex(index=SAMPLES, columns=REGIONS, fill_value=0)

spot_counts_group = pd.crosstab(
    adata_spot.obs["condition"],
    adata_spot.obs[REGION_KEY],
).reindex(index=["Young", "Aged", "AD"], columns=REGIONS, fill_value=0)

print("Sample × Region")
display(spot_counts_sample)

print("Condition × Region")
display(spot_counts_group)

spot_counts_sample.to_csv(
    OUT_DIR / "spot_counts_by_sample_region.tsv",
    sep="\t",
)

spot_counts_group.to_csv(
    OUT_DIR / "spot_counts_by_condition_region.tsv",
    sep="\t",
)

# 污染基因过滤

In [ ]:
# QC基因
EXCLUDE_GENES = [
    "Cox8b", "Actc1", "Acta1", "Myl7", "Tnni3",
    "Ckmt2", "Ckm", "Csrp3", "Hrc", "Tcap",
    "Klhl40", "Sln", "Pgam2",
    "Krt10", "Kap", "Aldob", "Spink1", "Htra4",
    "Miox", "Napsa", "Gkn3", "Car6", "Crisp3",
    "Pip", "Defb11", "Mucl2", "Scgb1b27", "Lct", "Tsacc",
    "Klk1", "Klk1b1", "Klk1b3", "Klk1b4", "Klk1b5",
    "Klk1b8", "Klk1b9", "Klk1b11", "Klk1b16",
    "Klk1b21", "Klk1b22", "Klk1b26", "Klk1b27",
    "Scgb2b26", "Scgb2b27", "Cd3d", "Cd3g", "Cox7a1",
    "Fxyd2", "Gm15543", "Ly6d", "S100g", "Trbc1",
    "Wfdc12", "Fabp4", "Fbp1",
    "Gh", "Guca2b", "Nppb", "Nppa", "Tnnc1",
    "Wfdc15b", "Dmkn", "Krt15",
    "Trbc2", "Wfdc18", "Cox6a2", "Krt8", "Krt18",
]

# 删除QC基因
before = adata_spot.n_vars

exclude_mask = adata_spot.var[SYMBOL_KEY].astype(str).isin(EXCLUDE_GENES)
removed = sorted(adata_spot.var.loc[exclude_mask, SYMBOL_KEY].astype(str).unique())

adata_spot = adata_spot[:, ~exclude_mask].copy()

print("删除前 genes:", before)
print("删除后 genes:", adata_spot.n_vars)
print("实际删除 genes:", len(removed))
print(removed)

# 直接跑 DEG

In [ ]:
# DEG
all_results = []

for region in tqdm(REGIONS, desc="Running regions"):

    for comparison, (test_group, reference_group) in CONTRASTS.items():

        mask = (
            adata_spot.obs[REGION_KEY].astype(str).eq(region)
            & adata_spot.obs["condition"].astype(str).isin([test_group, reference_group])
        )

        ad = adata_spot[mask].copy()

        test_mask = ad.obs["condition"].astype(str).eq(test_group).to_numpy()
        ref_mask = ad.obs["condition"].astype(str).eq(reference_group).to_numpy()

        n_test = test_mask.sum()
        n_ref = ref_mask.sum()

        if min(n_test, n_ref) < MIN_GROUP_SPOTS:
            continue

        # 基因检测过滤
        counts = sp.csr_matrix(ad.layers[COUNT_LAYER])

        detected_test = np.asarray((counts[test_mask] > 0).sum(axis=0)).ravel()
        detected_ref = np.asarray((counts[ref_mask] > 0).sum(axis=0)).ravel()

        keep = (
            (detected_test >= MIN_DETECTED_SPOTS)
            | (detected_ref >= MIN_DETECTED_SPOTS)
        )

        ad = ad[:, keep].copy()

        # Wilcoxon
        sc.tl.rank_genes_groups(
            ad,
            groupby="condition",
            groups=[test_group],
            reference=reference_group,
            method="wilcoxon",
            tie_correct=True,
            use_raw=False,
            n_genes=ad.n_vars,
        )

        result = sc.get.rank_genes_groups_df(
            ad,
            group=test_group,
        ).rename(columns={
            "names": "gene_id",
            "logfoldchanges": "log2FC",
            "pvals": "pvalue",
            "pvals_adj": "padj",
        })

        symbol_map = ad.var[SYMBOL_KEY].astype(str).to_dict()
        result["Symbol"] = result["gene_id"].astype(str).map(symbol_map)

        result.insert(0, "region", region)
        result.insert(1, "comparison", comparison)
        result.insert(2, "test_group", test_group)
        result.insert(3, "reference_group", reference_group)

        result = result.sort_values("padj").reset_index(drop=True)

        result.to_csv(
            PER_REGION_DIR / f"{region}__{comparison}.tsv.gz",
            sep="\t",
            index=False,
        )

        all_results.append(result)

        print(
            f"{region} | {comparison}: "
            f"{n_test:,} vs {n_ref:,} spots | "
            f"{ad.n_vars:,} genes"
        )

In [ ]:
# 合并总表
deg_all = pd.concat(all_results, ignore_index=True)

# 保存
deg_all.to_csv(
    OUT_DIR / "spot_DEG_all_regions_all_comparisons.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip",
)

print(deg_all.shape)
display(deg_all.head())

# 1、整体双向图

In [ ]:
# 参数
DEG_FILE = OUT_DIR / "spot_DEG_all_regions_all_comparisons.tsv.gz"

PADJ_THRESH = 0.01
LOG2FC_THRESH = 0.75

# 读取数据
deg = pd.read_csv(DEG_FILE, sep="\t")

print(deg.shape)
display(deg.head())

In [ ]:
# 筛选DEG
sig_deg = deg[
    (deg["padj"] < PADJ_THRESH) &
    (deg["log2FC"].abs() >= LOG2FC_THRESH)
].copy()

sig_deg["direction"] = np.where(
    sig_deg["log2FC"] > 0,
    "Increased",
    "Decreased"
)

# 统计数量
spot_deg_summary = (
    sig_deg
    .groupby(["region", "comparison", "direction"])["gene_id"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

display(spot_deg_summary)

print("Significant DEGs:", len(sig_deg))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 参数
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PLOT_REGIONS = [
    "CTX_1", "CTX_2", "CA1", "CA3", "DG",
    "TH", "HY", "STR", "FT", "VS"
]

COLORS = {
    ("AD_vs_Young", "Increased"): "#bd2126",
    ("Aged_vs_Young", "Increased"): "#fac7c4",
    ("AD_vs_Young", "Decreased"): "#1472b8",
    ("Aged_vs_Young", "Decreased"): "#b3c7e6",
}

LINTHRESH = 100
FIGSIZE = (8, 4.5)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# 整理数据
plot_df = (
    spot_deg_summary[
        spot_deg_summary["region"].isin(PLOT_REGIONS) &
        spot_deg_summary["comparison"].isin(["Aged_vs_Young", "AD_vs_Young"])
    ]
    .pivot(index="region", columns="comparison", values=["Increased", "Decreased"])
    .reindex(PLOT_REGIONS)
    .fillna(0)
)

x = np.arange(len(PLOT_REGIONS))
width = 0.34

ad_up = plot_df[("Increased", "AD_vs_Young")].values
ad_down = -plot_df[("Decreased", "AD_vs_Young")].values
aged_up = plot_df[("Increased", "Aged_vs_Young")].values
aged_down = -plot_df[("Decreased", "Aged_vs_Young")].values

# 画图
fig, ax = plt.subplots(figsize=FIGSIZE)

ax.bar(
    x - width / 2, ad_up, width,
    color=COLORS[("AD_vs_Young", "Increased")],
    label="AD vs Young: Increased"
)

ax.bar(
    x - width / 2, ad_down, width,
    color=COLORS[("AD_vs_Young", "Decreased")],
    label="AD vs Young: Decreased"
)

ax.bar(
    x + width / 2, aged_up, width,
    color=COLORS[("Aged_vs_Young", "Increased")],
    label="Aged vs Young: Increased"
)

ax.bar(
    x + width / 2, aged_down, width,
    color=COLORS[("Aged_vs_Young", "Decreased")],
    label="Aged vs Young: Decreased"
)

# y轴使用对称log
ax.set_yscale("symlog", linthresh=LINTHRESH)
ax.axhline(0, color="black", linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(PLOT_REGIONS, rotation=45, ha="right")

ax.set_ylabel("Number of differentially expressed genes")
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda y, pos: f"{abs(int(y))}")
)

ax.legend(
    frameon=False,
    fontsize=9,
    ncol=2,
)

ax.grid(False)

plt.tight_layout()

# 保存
plt.savefig(
    FIG_DIR / "spot_level_DEG_AD_Aged_diverging_symlog_padj0.01_log2FC0.75.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close()

In [ ]:
# ============================================================
# 各脑区 DEG 数量统计
# ============================================================

PADJ_THRESH = 0.01
LOG2FC_THRESH = 0.5

sig_deg = deg_clean[
    (deg_clean["padj"] < PADJ_THRESH) &
    (deg_clean["log2FC"].abs() >= LOG2FC_THRESH)
].copy()

sig_deg["direction"] = np.where(
    sig_deg["log2FC"] > 0,
    "Increased",
    "Decreased"
)

spot_deg_summary = (
    sig_deg
    .groupby(["region", "comparison", "direction"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["Increased", "Decreased"]:
    if col not in spot_deg_summary.columns:
        spot_deg_summary[col] = 0

display(spot_deg_summary)

In [ ]:
# ============================================================
# 各脑区 DEG 数量：AD / Aging 双柱双向图
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PLOT_REGIONS = [
    "CTX_1", "CTX_2", "HIP", "CA1", "CA3",
    "TH", "HY", "STR", "FT", "VS"
]

FIGSIZE = (8.0, 4.5)
BAR_WIDTH = 0.34

COLORS = {
    ("AD_vs_Young", "Increased"): "#bd2126",
    ("Aged_vs_Young", "Increased"): "#fac7c4",
    ("AD_vs_Young", "Decreased"): "#1472b8",
    ("Aged_vs_Young", "Decreased"): "#b3c7e6",
}

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

plot_df = (
    spot_deg_summary[
        spot_deg_summary["region"].isin(PLOT_REGIONS) &
        spot_deg_summary["comparison"].isin(["Aged_vs_Young", "AD_vs_Young"])
    ]
    .pivot(
        index="region",
        columns="comparison",
        values=["Increased", "Decreased"]
    )
    .reindex(PLOT_REGIONS)
    .fillna(0)
)

x = np.arange(len(PLOT_REGIONS))

ad_up = plot_df[("Increased", "AD_vs_Young")].to_numpy()
ad_down = -plot_df[("Decreased", "AD_vs_Young")].to_numpy()

aged_up = plot_df[("Increased", "Aged_vs_Young")].to_numpy()
aged_down = -plot_df[("Decreased", "Aged_vs_Young")].to_numpy()

fig, ax = plt.subplots(figsize=FIGSIZE)

ax.bar(
    x - BAR_WIDTH / 2, ad_up,
    width=BAR_WIDTH,
    color=COLORS[("AD_vs_Young", "Increased")],
    linewidth=0,
    label="AD vs Young: Increased"
)

ax.bar(
    x - BAR_WIDTH / 2, ad_down,
    width=BAR_WIDTH,
    color=COLORS[("AD_vs_Young", "Decreased")],
    linewidth=0,
    label="AD vs Young: Decreased"
)

ax.bar(
    x + BAR_WIDTH / 2, aged_up,
    width=BAR_WIDTH,
    color=COLORS[("Aged_vs_Young", "Increased")],
    linewidth=0,
    label="Aged vs Young: Increased"
)

ax.bar(
    x + BAR_WIDTH / 2, aged_down,
    width=BAR_WIDTH,
    color=COLORS[("Aged_vs_Young", "Decreased")],
    linewidth=0,
    label="Aged vs Young: Decreased"
)

ax.axhline(0, color="black", linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(PLOT_REGIONS, rotation=45, ha="right", fontsize=10)
ax.set_xlabel("Brain region", fontsize=11)

max_count = max(
    ad_up.max(),
    np.abs(ad_down).max(),
    aged_up.max(),
    np.abs(aged_down).max()
)

ylim = int(np.ceil(max_count / 20) * 20 + 20)

ax.set_ylim(-ylim, ylim)
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda value, pos: f"{abs(int(value))}")
)

ax.set_ylabel(
    "Number of differentially expressed genes",
    fontsize=11
)

ax.legend(
    frameon=False,
    fontsize=9,
    ncol=2,
    loc="upper right"
)

ax.grid(False)

for spine in ["top", "right", "bottom", "left"]:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_linewidth(0.8)

ax.tick_params(axis="both", width=0.8, labelsize=9)

save_path = (
    FIG_DIR /
    "spot_level_DEG_AD_Aged_diverging_padj0.01_log2FC0.5.pdf"
)

plt.tight_layout()
fig.savefig(
    save_path,
    bbox_inches="tight",
    transparent=True
)

plt.show()
plt.close()

print("保存至：", save_path)

# 2、双向富集分析

In [ ]:
# 前面先对每个脑区分别进行差异表达分析，比较包括 Aged vs Young 和 AD vs Young。差异基因筛选阈值统一设为 padj < 0.01 且 |log2FC| ≥ 0.75，然后按照表达变化方向分成 Up 和 Down 两组。

# 对每一个“脑区 × comparison × Up/Down”组合分别进行功能富集分析，数据库包括：

# GO Biological Process
# GO Molecular Function
# KEGG Mouse

# 富集分析时，background 不是直接使用全基因组，而是使用该脑区、该 comparison 中实际进入 Wilcoxon 差异检验的全部基因。这样可以保证背景基因和对应 DEG 的检测范围一致。

# 最终保留 Adjusted P-value < 0.05 的显著富集结果，形成这个总表：

# /histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/spot_level_DE_outer_min10/enrichment/DEG_enrichment_significant.tsv.gz

# 表中的主要字段含义如下：

# region：脑区
# comparison：比较组，例如 AD_vs_Young、Aged_vs_Young
# direction：差异基因方向，Up 或 Down
# database：富集数据库，GO_BP、GO_MF 或 KEGG
# n_input_genes：该组用于富集的显著 DEG 数量
# n_background_genes：该脑区该比较中实际参与差异检验的背景基因数量
# Term：富集到的通路或功能条目
# Overlap：输入 DEG 中有多少基因落在该通路中
# P-value：原始富集显著性
# Adjusted P-value：多重检验校正后的富集显著性
# Odds Ratio：富集强度
# Combined Score：Enrichr 综合评分
# Genes：落在该通路中的具体基因

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# 参数
OUT_DIR = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/spot_level_DE_outer_min10")
DEG_FILE = OUT_DIR / "spot_DEG_all_regions_all_comparisons.tsv.gz"
PADJ_THRESH = 0.01
LOG2FC_THRESH = 0.75
ENRICH_COMPARISONS = ["Aged_vs_Young", "AD_vs_Young"]

# 读取数据
deg = pd.read_csv(DEG_FILE, sep="\t")
print("DEG total:", deg.shape)
display(deg.head())

# 筛选DEG
sig_deg = deg[
    deg["comparison"].isin(ENRICH_COMPARISONS)
    & (deg["padj"] < PADJ_THRESH)
    & (deg["log2FC"].abs() >= LOG2FC_THRESH)
].copy()

sig_deg["direction"] = np.where(sig_deg["log2FC"] > 0, "Up", "Down")

# 查看结果
deg_counts = sig_deg.groupby(["region", "comparison", "direction"]).size().unstack(fill_value=0)

print("Significant DEGs:", len(sig_deg))
display(deg_counts)

In [ ]:
import gseapy as gp

# 参数
ENRICH_DIR = OUT_DIR / "enrichment"
ENRICH_DIR.mkdir(parents=True, exist_ok=True)

GENE_SETS = {
    "GO_BP": "GO_Biological_Process_2026",
    "GO_MF": "GO_Molecular_Function_2026",
    "KEGG": "KEGG_2019_Mouse",
}
MIN_GENES = 5
ENRICH_PADJ = 0.05

# 保存富集输入DEG
sig_deg.to_csv(ENRICH_DIR / "DEG_enrichment_gene_input.tsv.gz", sep="\t", index=False, compression="gzip")

# 读取富集数据库
libraries = {}
for db, lib_name in GENE_SETS.items():
    libraries[db] = gp.get_library(name=lib_name, organism="Mouse")
    print(f"{db}: {len(libraries[db])} pathways")

In [ ]:
# 参数
ENRICH_COMPARISONS = ["Aged_vs_Young", "AD_vs_Young"]
REGIONS = deg["region"].drop_duplicates().tolist()
MIN_GENES = 5
ENRICH_PADJ = 0.05

# 富集分析
enrich_results = []

for region in REGIONS:
    for comparison in ENRICH_COMPARISONS:
        bg_genes = deg[
            (deg["region"] == region) & (deg["comparison"] == comparison)
        ]["Symbol"].dropna().astype(str).unique().tolist()

        for direction in ["Up", "Down"]:
            genes = sig_deg[
                (sig_deg["region"] == region) &
                (sig_deg["comparison"] == comparison) &
                (sig_deg["direction"] == direction)
            ]["Symbol"].dropna().astype(str).unique().tolist()

            if len(genes) < MIN_GENES:
                print(f"{region} | {comparison} | {direction}: skipped ({len(genes)} genes)")
                continue

            for db, gene_set in libraries.items():
                enr = gp.enrichr(gene_list=genes, gene_sets=gene_set, background=bg_genes, outdir=None, no_plot=True)
                res = enr.results.copy()

                if len(res) == 0:
                    continue

                res.insert(0, "region", region)
                res.insert(1, "comparison", comparison)
                res.insert(2, "direction", direction)
                res.insert(3, "database", db)
                res.insert(4, "n_input_genes", len(genes))
                res.insert(5, "n_background_genes", len(bg_genes))
                enrich_results.append(res)

            print(f"{region} | {comparison} | {direction}: {len(genes)} genes")

# 合并结果
enrich_all = pd.concat(enrich_results, ignore_index=True)

# 保存全部富集结果
enrich_all.to_csv(
    ENRICH_DIR / "DEG_enrichment_all_results.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

# 保存显著富集结果
enrich_sig = enrich_all[enrich_all["Adjusted P-value"] < ENRICH_PADJ].copy()
enrich_sig = enrich_sig.sort_values(
    ["region", "comparison", "direction", "database", "Adjusted P-value"]
).reset_index(drop=True)

enrich_sig.to_csv(
    ENRICH_DIR / "DEG_enrichment_significant.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

print("All enrichment:", enrich_all.shape)
print("Significant enrichment:", enrich_sig.shape)
display(enrich_sig.head(20))

In [ ]:
# /histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/spot_level_DE_outer_min10/enrichment/DEG_enrichment_significant.tsv.gz

# 3、shared-only双向富集分析

# 先在每个脑区中分别定义 AD 特异、衰老特异和 AD/衰老共同变化的 DEG，再按上调/下调拆分，最后分别做 GO/KEGG ORA 富集。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# 参数
OUT_DIR = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/spot_level_DE_outer_min10")
DEG_FILE = OUT_DIR / "spot_DEG_all_regions_all_comparisons.tsv.gz"
GROUP_DIR = OUT_DIR / "enrichment_grouped"
GROUP_DIR.mkdir(parents=True, exist_ok=True)

PADJ_THRESH = 0.01
LOG2FC_THRESH = 0.75

# 读取数据
deg = pd.read_csv(DEG_FILE, sep="\t")
deg = deg[deg["comparison"].isin(["Aged_vs_Young", "AD_vs_Young"])].copy()

print(deg.shape)
display(deg.head())

In [ ]:
# 分开两个comparison
aged = deg[deg["comparison"] == "Aged_vs_Young"][["region", "gene_id", "Symbol", "log2FC", "padj"]].copy()
ad = deg[deg["comparison"] == "AD_vs_Young"][["region", "gene_id", "Symbol", "log2FC", "padj"]].copy()

aged = aged.rename(columns={"Symbol": "Symbol_aged", "log2FC": "log2FC_aged", "padj": "padj_aged"})
ad = ad.rename(columns={"Symbol": "Symbol_ad", "log2FC": "log2FC_ad", "padj": "padj_ad"})

# 两个comparison都进入检验的基因
grouped = aged.merge(ad, on=["region", "gene_id"], how="inner")
grouped["Symbol"] = grouped["Symbol_aged"].fillna(grouped["Symbol_ad"])

print("Common tested genes:", grouped.shape)
display(grouped.head())

In [ ]:
# 是否显著
grouped["aged_sig"] = (grouped["padj_aged"] < PADJ_THRESH) & (grouped["log2FC_aged"].abs() >= LOG2FC_THRESH)
grouped["ad_sig"] = (grouped["padj_ad"] < PADJ_THRESH) & (grouped["log2FC_ad"].abs() >= LOG2FC_THRESH)

# 初始化
grouped["DEG_group"] = "Not_sig"
grouped["direction"] = pd.NA

# Aged only
mask = grouped["aged_sig"] & ~grouped["ad_sig"]
grouped.loc[mask, "DEG_group"] = "Aged_only"
grouped.loc[mask, "direction"] = np.where(grouped.loc[mask, "log2FC_aged"] > 0, "Up", "Down")

# AD only
mask = grouped["ad_sig"] & ~grouped["aged_sig"]
grouped.loc[mask, "DEG_group"] = "AD_only"
grouped.loc[mask, "direction"] = np.where(grouped.loc[mask, "log2FC_ad"] > 0, "Up", "Down")

# 两边都显著
both = grouped["aged_sig"] & grouped["ad_sig"]
same_up = both & (grouped["log2FC_aged"] > 0) & (grouped["log2FC_ad"] > 0)
same_down = both & (grouped["log2FC_aged"] < 0) & (grouped["log2FC_ad"] < 0)
discordant = both & ~(same_up | same_down)

grouped.loc[same_up | same_down, "DEG_group"] = "Shared"
grouped.loc[same_up, "direction"] = "Up"
grouped.loc[same_down, "direction"] = "Down"
grouped.loc[discordant, "DEG_group"] = "Discordant"

# 查看数量
display(
    grouped[grouped["DEG_group"].isin(["AD_only", "Shared", "Aged_only"])]
    .groupby(["region", "DEG_group", "direction"])
    .size()
    .unstack(fill_value=0)
)

print("Discordant:", (grouped["DEG_group"] == "Discordant").sum())

In [ ]:
# 富集输入
grouped_deg = grouped[
    grouped["DEG_group"].isin(["AD_only", "Shared", "Aged_only"])
].copy()

# 每个脑区共同tested genes作为背景
background = grouped[["region", "gene_id", "Symbol"]].drop_duplicates()

grouped_deg.to_csv(GROUP_DIR / "DEG_grouped_input.tsv.gz", sep="\t", index=False, compression="gzip")
background.to_csv(GROUP_DIR / "DEG_grouped_background.tsv.gz", sep="\t", index=False, compression="gzip")

print("Grouped DEG:", grouped_deg.shape)
print("Background:", background.shape)

display(
    background.groupby("region")["gene_id"]
    .nunique()
    .to_frame("n_background_genes")
)

In [ ]:
import gseapy as gp

# 参数
GENE_SETS = {
    "GO_BP": "GO_Biological_Process_2026",
    "GO_MF": "GO_Molecular_Function_2026",
    "KEGG": "KEGG_2019_Mouse",
}
MIN_GENES = 5
ENRICH_PADJ = 0.05

# 读取数据库
libraries = {}
for db, lib_name in GENE_SETS.items():
    libraries[db] = gp.get_library(name=lib_name, organism="Mouse")
    print(f"{db}: {len(libraries[db])} pathways")

In [ ]:
# 参数
DEG_GROUPS = ["AD_only", "Shared", "Aged_only"]
DIRECTIONS = ["Up", "Down"]

# 富集分析
enrich_results = []

for region in grouped_deg["region"].drop_duplicates():
    bg_genes = background[background["region"] == region]["Symbol"].dropna().astype(str).unique().tolist()

    for deg_group in DEG_GROUPS:
        for direction in DIRECTIONS:
            genes = grouped_deg[
                (grouped_deg["region"] == region) &
                (grouped_deg["DEG_group"] == deg_group) &
                (grouped_deg["direction"] == direction)
            ]["Symbol"].dropna().astype(str).unique().tolist()

            if len(genes) < MIN_GENES:
                print(f"{region} | {deg_group} | {direction}: skipped ({len(genes)} genes)")
                continue

            for db, gene_set in libraries.items():
                enr = gp.enrichr(gene_list=genes, gene_sets=gene_set, background=bg_genes, outdir=None, no_plot=True)
                res = enr.results.copy()

                if len(res) == 0:
                    continue

                res.insert(0, "region", region)
                res.insert(1, "DEG_group", deg_group)
                res.insert(2, "direction", direction)
                res.insert(3, "database", db)
                res.insert(4, "n_input_genes", len(genes))
                res.insert(5, "n_background_genes", len(bg_genes))
                enrich_results.append(res)

            print(f"{region} | {deg_group} | {direction}: {len(genes)} genes")

In [ ]:
# 合并结果
enrich_all = pd.concat(enrich_results, ignore_index=True)

# 保存全部结果
enrich_all.to_csv(
    GROUP_DIR / "DEG_grouped_enrichment_all_results.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

# 显著结果
enrich_sig = enrich_all[enrich_all["Adjusted P-value"] < ENRICH_PADJ].copy()
enrich_sig = enrich_sig.sort_values(
    ["region", "DEG_group", "direction", "database", "Adjusted P-value"]
).reset_index(drop=True)

enrich_sig.to_csv(
    GROUP_DIR / "DEG_grouped_enrichment_significant.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

print("All enrichment:", enrich_all.shape)
print("Significant enrichment:", enrich_sig.shape)
display(enrich_sig.head(20))

# 4、gsea分析

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gseapy as gp
from tqdm.auto import tqdm
from IPython.display import display

# 参数
OUT_DIR = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/spot_level_DE_outer_min10")
DEG_FILE = OUT_DIR / "spot_DEG_all_regions_all_comparisons.tsv.gz"
GSEA_DIR = OUT_DIR / "gsea"
GSEA_DIR.mkdir(parents=True, exist_ok=True)

COMPARISONS = ["Aged_vs_Young", "AD_vs_Young"]
GENE_SETS = {
    "GO_BP": "GO_Biological_Process_2026",
    "GO_MF": "GO_Molecular_Function_2026",
    "KEGG": "KEGG_2019_Mouse",
}

# 读取数据
deg = pd.read_csv(DEG_FILE, sep="\t")

# 读取数据库
libraries = {}
for db, lib_name in GENE_SETS.items():
    libraries[db] = gp.get_library(name=lib_name, organism="Mouse")
    print(f"{db}: {len(libraries[db])} pathways")

print(deg.shape)
display(deg.head())

In [ ]:
# 测试一个脑区
region = "CA1"
comparison = "AD_vs_Young"

rank_df = deg[
    (deg["region"] == region) &
    (deg["comparison"] == comparison)
][["Symbol", "scores"]].dropna().copy()

# 同名Symbol只保留|score|最大的一个
rank_df["abs_score"] = rank_df["scores"].abs()
rank_df = rank_df.sort_values("abs_score", ascending=False).drop_duplicates("Symbol")
rank_df = rank_df.sort_values("scores", ascending=False)[["Symbol", "scores"]]

print(rank_df.shape)
display(rank_df.head())
display(rank_df.tail())

In [ ]:
# 测试GSEA
pre_res = gp.prerank(
    rnk=rank_df,
    gene_sets=libraries["GO_BP"],
    min_size=10,
    max_size=500,
    permutation_num=1000,
    seed=0,
    threads=4,
    outdir=None,
    verbose=False,
)

gsea_test = pre_res.res2d.reset_index()

print(gsea_test.shape)
display(gsea_test.head(20))

In [ ]:
# 参数
REGIONS = ["CTX_1", "CTX_2", "CTX_3", "HIP", "CA1", "CA3", "DG", "TH", "HY", "STR", "FT", "VS"]
COMPARISONS = ["Aged_vs_Young", "AD_vs_Young"]
MIN_SIZE = 10
MAX_SIZE = 500
PERMUTATION_NUM = 1000
GSEA_FDR = 0.05

# 批量GSEA
gsea_results = []

for region in tqdm(REGIONS, desc="Running GSEA"):
    for comparison in COMPARISONS:
        rank_df = deg[
            (deg["region"] == region) & (deg["comparison"] == comparison)
        ][["Symbol", "scores", "log2FC"]].dropna().copy()

        # 一个Symbol只保留|score|最大的gene，并用log2FC轻微打破score并列
        rank_df["abs_score"] = rank_df["scores"].abs()
        rank_df = rank_df.sort_values("abs_score", ascending=False).drop_duplicates("Symbol")
        rank_df["rank_score"] = rank_df["scores"] + rank_df["log2FC"] * 1e-9
        rank_df = rank_df.sort_values("rank_score", ascending=False)[["Symbol", "rank_score"]]

        for db, gene_set in libraries.items():
            pre_res = gp.prerank(
                rnk=rank_df, gene_sets=gene_set,
                min_size=MIN_SIZE, max_size=MAX_SIZE,
                permutation_num=PERMUTATION_NUM, seed=0,
                threads=4, outdir=None, verbose=False
            )

            res = pre_res.res2d.reset_index(drop=True)
            res.insert(0, "region", region)
            res.insert(1, "comparison", comparison)
            res.insert(2, "database", db)
            res.insert(3, "n_ranked_genes", len(rank_df))
            gsea_results.append(res)

        print(f"{region} | {comparison}: {len(rank_df):,} genes")

In [ ]:
# 合并结果
gsea_all = pd.concat(gsea_results, ignore_index=True)

# 数值列
for col in ["ES", "NES", "NOM p-val", "FDR q-val", "FWER p-val"]:
    if col in gsea_all.columns:
        gsea_all[col] = pd.to_numeric(gsea_all[col], errors="coerce")

# 保存全部结果
gsea_all.to_csv(GSEA_DIR / "GSEA_all_regions_all_results.tsv.gz", sep="\t", index=False, compression="gzip")

# 保存显著结果
gsea_sig = gsea_all[gsea_all["FDR q-val"] < GSEA_FDR].copy()
gsea_sig = gsea_sig.sort_values(["region", "comparison", "database", "FDR q-val", "NES"], ascending=[True, True, True, True, False])
gsea_sig.to_csv(GSEA_DIR / "GSEA_all_regions_significant.tsv.gz", sep="\t", index=False, compression="gzip")

print("All GSEA:", gsea_all.shape)
print("Significant GSEA:", gsea_sig.shape)
display(gsea_sig.head(20))

## gsea绘图

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# 参数
GSEA_DIR = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/spot_level_DE_outer_min10/gsea")
GSEA_FILE = GSEA_DIR / "GSEA_all_regions_all_results.tsv.gz"
FIG_DIR = GSEA_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATABASE = "GO_BP"
FDR_THRESH = 0.05
TOP_PATHWAYS = 25
MIN_SIGNIFICANT_GROUPS = 2

PLOT_REGIONS = ["CTX_1", "CTX_2", "CTX_3", "HIP", "CA1", "CA3", "DG", "TH", "HY", "STR", "FT", "VS"]
COMPARISONS = ["Aged_vs_Young", "AD_vs_Young"]

COLORS = ["#3288bd", "#66c2a5", "#ffffbf", "#f46d43", "#d53e4f"]
FIGSIZE = (13, 10)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# 读取数据
df = pd.read_csv(GSEA_FILE, sep="\t")
df = df[
    (df["database"] == DATABASE) &
    (df["region"].isin(PLOT_REGIONS)) &
    (df["comparison"].isin(COMPARISONS))
].copy()

df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")

# 选择代表性通路
sig = df[df["FDR q-val"] < FDR_THRESH].copy()

pathway_rank = (
    sig.groupby("Term")
    .agg(
        n_significant=("Term", "size"),
        min_fdr=("FDR q-val", "min"),
        max_abs_nes=("NES", lambda x: x.abs().max()),
    )
    .reset_index()
)

pathway_rank = pathway_rank[
    pathway_rank["n_significant"] >= MIN_SIGNIFICANT_GROUPS
].sort_values(
    ["n_significant", "min_fdr", "max_abs_nes"],
    ascending=[False, True, False],
)

selected_terms = pathway_rank.head(TOP_PATHWAYS)["Term"].tolist()

plot_df = df[df["Term"].isin(selected_terms)].copy()

# X轴顺序
x_order = [(region, comparison) for region in PLOT_REGIONS for comparison in COMPARISONS]
x_map = {x: i for i, x in enumerate(x_order)}
y_map = {term: i for i, term in enumerate(selected_terms[::-1])}

plot_df["x"] = [x_map[(r, c)] for r, c in zip(plot_df["region"], plot_df["comparison"])]
plot_df["y"] = plot_df["Term"].map(y_map)

# 气泡大小：-log10(FDR)
plot_df["FDR_plot"] = plot_df["FDR q-val"].clip(lower=1e-6)
plot_df["significance"] = -np.log10(plot_df["FDR_plot"])
plot_df["size"] = 15 + plot_df["significance"].clip(upper=6) * 18

# 颜色
cmap = LinearSegmentedColormap.from_list("NES", COLORS, N=100)
max_nes = np.nanmax(np.abs(plot_df["NES"]))
norm = TwoSlopeNorm(vmin=-max_nes, vcenter=0, vmax=max_nes)

# 画图
fig, ax = plt.subplots(figsize=FIGSIZE)

# 脑区交替背景
for i, region in enumerate(PLOT_REGIONS):
    start = i * len(COMPARISONS) - 0.5
    end = start + len(COMPARISONS)

    if i % 2 == 0:
        ax.axvspan(start, end, color="grey", alpha=0.10, zorder=0)

sc = ax.scatter(
    plot_df["x"], plot_df["y"],
    c=plot_df["NES"],
    s=plot_df["size"],
    cmap=cmap, norm=norm,
    edgecolors="none",
    zorder=2,
)

# 坐标轴
ax.set_xticks(range(len(x_order)))
ax.set_xticklabels(
    ["Aged vs Young" if c == "Aged_vs_Young" else "AD vs Young" for _, c in x_order],
    rotation=90,
)

ax.set_yticks(range(len(selected_terms)))
ax.set_yticklabels(selected_terms[::-1])

ax.set_xlim(-0.5, len(x_order) - 0.5)
ax.set_ylim(-0.5, len(selected_terms) - 0.5)
ax.grid(False)

# 顶部脑区标签
ax_top = ax.secondary_xaxis("top")
region_centers = [i * len(COMPARISONS) + (len(COMPARISONS) - 1) / 2 for i in range(len(PLOT_REGIONS))]
ax_top.set_xticks(region_centers)
ax_top.set_xticklabels(PLOT_REGIONS)
ax_top.tick_params(length=0, pad=6)

# NES图例
cbar = fig.colorbar(sc, ax=ax, pad=0.015)
cbar.set_label("NES")

# Size图例
size_values = [1, 2, 3, 4]
handles = [
    ax.scatter([], [], s=15 + v * 18, color="grey", edgecolors="none")
    for v in size_values
]
ax.legend(
    handles,
    [f"{v}" for v in size_values],
    title=r"$-\log_{10}$(FDR)",
    frameon=False,
    bbox_to_anchor=(1.13, 0.5),
    loc="center left",
)

ax.set_xlabel("")
ax.set_ylabel("GSEA pathway")

plt.tight_layout()

# 保存
plt.savefig(
    FIG_DIR / f"GSEA_{DATABASE}_multi_region_bubble_heatmap.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close()

# 3、Igh

In [ ]:
import scanpy as sc
from pathlib import Path

# 参数
INPUT_DIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/anno_h5ads")
SAMPLES = ["G21-4", "G21-7", "G21-1", "G21-8", "G21-2", "G21-5"]
SYMBOL_KEY = "Symbol"

# 读取一个样本检查基因
ad = sc.read_h5ad(INPUT_DIR / "G21-4_regions.h5ad")

symbols = ad.var[SYMBOL_KEY].astype(str)
igh_genes = sorted(symbols[symbols.str.startswith("Igh")].unique())

print(igh_genes)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from pathlib import Path

# 参数
INPUT_DIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/anno_h5ads")
SAMPLES = ["G21-4", "G21-7", "G21-1", "G21-8", "G21-2", "G21-5"]
GROUP_MAP = {"G21-4":"Young", "G21-7":"Young", "G21-1":"Aged", "G21-8":"Aged", "G21-2":"AD", "G21-5":"AD"}

GENES = ["Ighd", "Ighg2c", "Ighm", "Igkc"]
SPOT_SIZE = 23

# 读取数据
ads = {}
for sample in SAMPLES:
    ads[sample] = sc.read_h5ad(INPUT_DIR / f"{sample}_regions.h5ad")

# 画图
for gene in GENES:
    fig, axes = plt.subplots(2, 3, figsize=(10, 6))

    for ax, sample in zip(axes.ravel(), SAMPLES):
        ad = ads[sample]
        idx = ad.var_names[ad.var["Symbol"].astype(str) == gene]

        if len(idx) == 0:
            ax.set_title(f"{sample}\n{gene} not found")
            ax.axis("off")
            continue

        sc.pl.spatial(
            ad,
            color=idx[0],
            layer="counts",
            img_key="hires",
            bw=True,
            alpha_img=0.5,
            spot_size=SPOT_SIZE,
            cmap="Reds",
            frameon=False,
            title=f"{sample} ({GROUP_MAP[sample]})",
            ax=ax,
            show=False,
        )
        ax.grid(False)

    fig.suptitle(f"{gene} | raw counts", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 参数
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

INPUT_DIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/anno_h5ads")

SAMPLES = ["G21-4", "G21-7", "G21-1", "G21-8", "G21-2", "G21-5"]
GROUP_MAP = {"G21-4":"Young", "G21-7":"Young", "G21-1":"Aged", "G21-8":"Aged", "G21-2":"AD", "G21-5":"AD"}

REGIONS = ["CTX_1", "CTX_2", "CTX_3", "HIP", "CA1", "CA3", "DG", "TH", "HY", "STR", "FT", "VS"]

REGION_KEY = "Regions"
COUNT_LAYER = "counts"
SYMBOL_KEY = "Symbol"

GENES = ["Ighm", "Ighd", "Ighg2c", "Igkc"]
TARGET_SUM = 1e4
figsize = (16, 5)

# 读取数据 + 计算 sample x region 平均表达
rows = []

for sample in SAMPLES:
    ad = sc.read_h5ad(INPUT_DIR / f"{sample}_regions.h5ad")
    ad = ad[ad.obs[REGION_KEY].astype(str).isin(REGIONS)].copy()
    ad.X = ad.layers[COUNT_LAYER].copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)

    for region in REGIONS:
        sub = ad[ad.obs[REGION_KEY].astype(str) == region].copy()

        row = {"sample": sample, "group": GROUP_MAP[sample], "region": region}

        for gene in GENES:
            idx = sub.var_names[sub.var[SYMBOL_KEY].astype(str) == gene]
            if len(idx) == 0 or sub.n_obs == 0:
                row[gene] = np.nan
            else:
                x = sub[:, idx[0]].X
                row[gene] = np.asarray(x.mean()).item()

        rows.append(row)

df = pd.DataFrame(rows)

# 如果想把文章定义的 Igh 也一起画上，取消下面两行注释
# df["Igh"] = df[["Ighd", "Ighg2c", "Ighm"]].max(axis=1)
# GENES = ["Ighm", "Ighd", "Ighg2c", "Igkc", "Igh"]

# 按基因做 0-1 标准化，得到 Relative expression
df_plot = df.copy()

for gene in GENES:
    v = df_plot[gene]
    vmin, vmax = v.min(), v.max()
    if pd.isna(vmin) or pd.isna(vmax) or vmax == vmin:
        df_plot[gene] = 0
    else:
        df_plot[gene] = (v - vmin) / (vmax - vmin)

# 画图
fig, axes = plt.subplots(2, 3, figsize=figsize)
axes = axes.ravel()

for i, sample in enumerate(SAMPLES):
    ax = axes[i]

    mat = df_plot[df_plot["sample"] == sample].set_index("region")[GENES].T
    mat = mat[REGIONS]

    sns.heatmap(
        mat,
        ax=ax,
        cmap="Spectral_r",
        vmin=0,
        vmax=1,
        cbar=(i == len(SAMPLES) - 1),
        cbar_kws={"label": "Relative expression"},
        linewidths=0.3,
        linecolor="gray"
    )

    ax.set_title(f"{sample} ({GROUP_MAP[sample]})", fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11, fontstyle="italic")
    ax.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 参数
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

INPUT_DIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/anno_h5ads")

SAMPLES = ["G21-4", "G21-1", "G21-2"]
GROUP_MAP = {"G21-4":"Young", "G21-1":"Aged", "G21-2":"AD"}

REGIONS = ["CTX_1", "CTX_2", "CTX_3", "HIP", "CA1", "CA3", "DG", "TH", "HY", "STR", "FT", "VS"]

REGION_KEY = "Regions"
COUNT_LAYER = "counts"
SYMBOL_KEY = "Symbol"
GENE = "Ighm"

TARGET_SUM = 1e4
figsize = (10, 2.8)

# 读取数据 + 计算每个样本每个脑区的平均表达
rows = []

for sample in SAMPLES:
    ad = sc.read_h5ad(INPUT_DIR / f"{sample}_regions.h5ad")
    ad = ad[ad.obs[REGION_KEY].astype(str).isin(REGIONS)].copy()
    ad.X = ad.layers[COUNT_LAYER].copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)

    idx = ad.var_names[ad.var[SYMBOL_KEY].astype(str) == GENE]
    if len(idx) == 0:
        continue
    gene_id = idx[0]

    for region in REGIONS:
        sub = ad[ad.obs[REGION_KEY].astype(str) == region]
        if sub.n_obs == 0:
            value = np.nan
        else:
            x = sub[:, gene_id].X
            value = np.asarray(x.mean()).item()
        rows.append({
            "sample": f"{sample} ({GROUP_MAP[sample]})",
            "region": region,
            "expr": value
        })

df = pd.DataFrame(rows)

# 转成热图矩阵
mat = df.pivot(index="sample", columns="region", values="expr").loc[
    [f"{s} ({GROUP_MAP[s]})" for s in SAMPLES], REGIONS
]

# 0-1相对表达，更像文章那种 Relative expression
v = mat.values
vmin = np.nanmin(v)
vmax = np.nanmax(v)
mat_plot = (mat - vmin) / (vmax - vmin) if vmax > vmin else mat * 0

# 画图
fig, ax = plt.subplots(figsize=figsize)

sns.heatmap(
    mat_plot,
    cmap="Spectral_r",
    vmin=0, vmax=1,
    linewidths=0.3,
    linecolor="gray",
    cbar_kws={"label": "Relative expression"},
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
ax.grid(False)

plt.title("Ighm", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 参数
INPUT_DIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/anno_h5ads")
SAMPLES = ["G21-4", "G21-1", "G21-2"]
GROUP_MAP = {"G21-4":"Young", "G21-1":"Aged", "G21-2":"AD"}
REGIONS = ["CTX_1", "CTX_2", "CTX_3", "HIP", "CA1", "CA3", "DG", "TH", "HY", "STR", "FT", "VS"]

GENE = "Ighm"
REGION_KEY = "Regions"
SYMBOL_KEY = "Symbol"
figsize = (8, 2.5)

plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# 计算脑区平均表达
rows = []

for sample in SAMPLES:
    ad = sc.read_h5ad(INPUT_DIR / f"{sample}_regions.h5ad")
    gene_id = ad.var_names[ad.var[SYMBOL_KEY].astype(str) == GENE][0]

    for region in REGIONS:
        sub = ad[ad.obs[REGION_KEY].astype(str) == region]
        x = sub[:, gene_id].X
        value = np.asarray(x.mean()).item()

        rows.append({
            "sample": f"{sample} ({GROUP_MAP[sample]})",
            "region": region,
            "expression": value
        })

df = pd.DataFrame(rows)

mat = df.pivot(index="sample", columns="region", values="expression")
mat = mat.loc[[f"{s} ({GROUP_MAP[s]})" for s in SAMPLES], REGIONS]

# 画图
fig, ax = plt.subplots(figsize=figsize)

sns.heatmap(
    mat,
    cmap="Reds",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Mean expression"},
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title(GENE, fontsize=12, fontstyle="italic")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
ax.grid(False)

plt.tight_layout()
plt.show()

# 基因可视化

In [ ]:
import scanpy as sc

In [ ]:
ad1=sc.read("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260210_2/pp_data/G21-4_repaired_pp.h5ad")
ad1

In [ ]:
sc.pl.spatial(ad1,color=["Mrps10", "Bin1", "Mbp", "Sorcs1", "Ncam1", "Cd74", "Dctn6", "Nrn1", "C4b"],gene_symbols='Symbol',cmap="Spectral_r",spot_size=25,layer='counts')

In [ ]:
sc.pl.spatial(ad1,color=["C4b","C3","Ptpra"],gene_symbols='Symbol',cmap="Spectral_r",spot_size=25,layer='counts')

In [ ]:
sc.pl.spatial(ad1,color=["Myl6", "Pkm", "Klc1","Bin1"],gene_symbols='Symbol',cmap="Spectral_r",spot_size=25,layer='counts')

In [ ]:
ad4=sc.read("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/anno_h5ads/G21-1_regions.h5ad")
ad4

In [ ]:
sc.pl.spatial(ad4,color=["Myl6", "Pkm", "Klc1","Bin1"],gene_symbols='Symbol',cmap="Spectral_r",spot_size=25,layer='counts')

In [ ]:
# 参数
gene = "Bin1"

# 找到对应的 var_name
gene_id = ad4.var.index[ad4.var["Symbol"] == gene][0]

# 画小提琴图
sc.pl.violin(
    ad4,
    keys=gene_id,
    groupby="Regions",
    layer="counts",
    stripplot=False,
    rotation=90,
)

In [ ]:
gene = "Bin1"
gene_id = ad4.var.index[ad4.var["Symbol"] == gene][0]

sc.pl.violin(
    ad4,
    keys=gene_id,
    stripplot=False,
)

In [ ]:
ad_vs = ad4[ad4.obs["Regions"] == "VS"].copy()

gene_id = ad_vs.var.index[ad_vs.var["Symbol"] == "Bin1"][0]

sc.pl.violin(
    ad_vs,
    keys=gene_id,
    stripplot=False,layer="counts",
)

# 000